# User-defined Turbine Operation Models

Beginning in v4.x, FLORIS supports user-defined turbine operation models that can be passed directly into FLORIS using `fmodel.set_operation_model`. A user-defined operation model may be a dynamic or static class. If the operation model is a dynamic class, it must conform to the `attrs` package for declaring attributes. Additionally all user-defined operation models should inherit from the abstract parent class `BaseOperationModel`, available in FLORIS.

All operation models must implement the following "fundamental" methods:
- `power`: computes the power output of the turbine in Watts
- `thrust_coefficient`: computes the dimensionless thrust coefficient of the turbine
- `axial_induction`: computes the dimensionless axial induction factor of the turbine

Operation models may then implement additional methods as needed.

The following arguments are passed to the operation model fundamental methods at runtime:

| Argument | Data type | Description |
|----------|-----------|----------|
| `power_thrust_table` | `dict` | Dictionary of model parameters defined on the turbine input yaml |
| `velocities` | `NDArrayFloat` | Array of inflow velocities (in m/s) to each turbine grid point, dimensions `(n_findex, n_turbines, n_grid, n_grid)` |
| `turbulence_intensities` | `NDArrayFloat` | Array of inflow turbulence intensities (as decimal values) to each turbine, dimensions `(n_findex, n_turbines, 1, 1)` |
| `air_density` | `float` | Ambient air density in kg/m^3 |
| `yaw_angles` | `NDArrayFloat` | Array of turbine yaw angles (in degrees, as misalignments from the inflow wind direction), dimensions `(n_findex, n_turbines)` |
| `tilt_angles` | `NDArrayFloat` | Array of turbine absolute [CHECK] tilt angles (in degrees, positive means tilted backwards), dimensions `(n_findex, n_turbines)` |
| `power_setpoints` |  `NDArrayFloat` | Array of turbine power setpoints (in Watts), dimensions `(n_findex, n_turbines)` |
| `awc_modes` | `NDArrayStr` | Array of strings specifying the AWC mode for each turbine, dimensions `(n_findex, n_turbines)` |
| `awc_amplitudes` | `NDArrayFloat` | Array of AWC amplitudes (in degrees) for each turbine, dimensions `(n_findex, n_turbines)` |
| `tilt_interp` | `interpolator` | Scipy 1D interpolator to find the (floating) tilt angle as a function of wind speed |
| `average_method` | `string` | Averaging method for combining velocities over the turbine grid points |
| `cubature_weights` | `NDArrayFloat` | Weights for cubature grid computation of rotor-effective velocity, dimensions `(1, n_grid x n_grid)`|
| `correct_cp_ct_for_tilt` | `NDArrayInt` | Flag for correcting power and thrust curves to account for platform tilt, dimensions `(n_findex, n_turbines)` |
| `**_` | -- | Catch-all for unused arguments |

Not all of these arguments must be used or defined as arguments by the user, as long as the final argument be `**_` to allow for unused arguments.

Each of the fundamental methods must return an array of floats (`NDArrayFloat`) with dimensions `(n_findex, n_turbines)`, representing the compute power, thrust coefficient, or axial induction factor for each turbine at each flow condition index.

### Static example

We begin with a very simple example that will produce a constant power, thrust coefficient, and axial induction factor regardless of the inputs. We are using a static class for this example; this class does not need to be instantiated and has no attributes.

In [ ]:
import numpy as np
from attrs import define, field
from floris.type_dec import floris_float_type, NDArrayFloat
from floris.core.turbine.operation_models import BaseOperationModel

@define
class ConstantValueTurbine(BaseOperationModel):
    """
    A simple turbine operation model that returns constant values for power,
    thrust coefficient, and axial induction factor regardless of input conditions.
    """
    @staticmethod
    def power(
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
         # Constant power of 500 kW, in correct shape (n_findex, n_turbines)
        return 500000.0 * np.ones(velocities.shape[0:2], dtype=floris_float_type)

    @staticmethod
    def thrust_coefficient(
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        # Return thrust coefficient based on actuator disk theory
        # Because the class is static, we can call the axial_induction method directly
        a = ConstantValueTurbine.axial_induction(velocities)
        return 4 * a * (1 - a)

    @staticmethod
    def axial_induction(
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        # Constant axial induction factor of 0.3
        return 0.3 * np.ones(velocities.shape[0:2], dtype=floris_float_type)

Let's now use this constant operation model in FLORIS.

In [ ]:
from floris import FlorisModel, TimeSeries

fmodel = FlorisModel("defaults")
time_series = TimeSeries(
    wind_directions=np.array([270.0, 270.0, 280.0]),
    wind_speeds=np.array([8.0, 10.0, 12.0]),
    turbulence_intensities=np.array([0.06, 0.06, 0.06]),
)
fmodel.set(
    layout_x = [0.0, 500.0],
    layout_y = [0.0, 0.0],
    wind_data=time_series,
)
fmodel.set_operation_model(ConstantValueTurbine)

fmodel.run()

print("Powers [W]:\n", fmodel.get_turbine_powers(), "\n")
print("Thrust coefficients [-]:\n", fmodel.get_turbine_thrust_coefficients(), "\n")
print("Axial induction factors [-]:\n", fmodel.get_turbine_axial_induction_factors(), "\n")

## Dynamic example

Now, we will create an operation model that allows the user to set attributes at instantiation. In this example, we will create an operation model that allows the user to set constant power, thrust coefficient, and axial induction factor values at instantiation. These values will then be returned by the fundamental methods regardless of the inputs. We use the `attrs` package to define attributes of the class.

In [ ]:
@define
class DynamicValueTurbine(BaseOperationModel):
    """
    A simple turbine operation model that returns constant values for power,
    thrust coefficient, and axial induction factor regardless of input conditions,
    based on user-defined attributes.
    """
    power_value = field(init=True, default=600000.0, type=floris_float_type)
    axial_induction_value = field(init=True, default=0.2, type=floris_float_type)
    def power(
        self,
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
         # Constant power of 500 kW, in correct shape (n_findex, n_turbines)
        return self.power_value * np.ones(velocities.shape[0:2], dtype=floris_float_type)

    def thrust_coefficient(
        self,
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        # Return thrust coefficient based on actuator disk theory
        # Because the class is static, we can call the axial_induction method directly
        a = self.axial_induction(velocities)
        return 4 * a * (1 - a)

    def axial_induction(
        self,
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        # Constant axial induction factor of 0.3
        return self.axial_induction_value * np.ones(velocities.shape[0:2], dtype=floris_float_type)

To use this class, we must first instantiate it. If we instantiate it without any arguments, the default values will be used. Otherwise, we can pass in our desired constant values.

In [ ]:
turbine_operation_model = DynamicValueTurbine()
fmodel.set_operation_model(turbine_operation_model)
fmodel.run()

print("Powers [W]:\n", fmodel.get_turbine_powers(), "\n")
print("Thrust coefficients [-]:\n", fmodel.get_turbine_thrust_coefficients(), "\n")
print("Axial induction factors [-]:\n", fmodel.get_turbine_axial_induction_factors(), "\n")

In [ ]:
turbine_operation_model = DynamicValueTurbine(power_value=750000.0, axial_induction_value=0.25)
fmodel.set_operation_model(turbine_operation_model)
fmodel.run()

print("Powers [W]:\n", fmodel.get_turbine_powers(), "\n")
print("Thrust coefficients [-]:\n", fmodel.get_turbine_thrust_coefficients(), "\n")
print("Axial induction factors [-]:\n", fmodel.get_turbine_axial_induction_factors(), "\n")

## More complex example

Now, let's use an example where some parameters are defined on the `power_thrust_table` on the turbine input yaml, and some parameters are set upon instantiation of the class.

In [ ]:
from floris.core.turbine import SimpleTurbine

@define
class ScaledTurbine(BaseOperationModel):
    """
    A turbine operation model that scales power and thrust coefficient
    based on a user-defined scaling factor. This will use methods from the
    prepackaged SimpleTurbine model, leaving some values as default.

    Scaling only applies to power, not to thrust_coefficient or
    axial_induction. We also demonstrate that other "nonfundamental" methods
    can be used on the class.
    """
    scaling_factor = field(init=True, default=1.0, type=floris_float_type)

    def power(
        self,
        power_thrust_table: dict,
        velocities: NDArrayFloat,
        air_density: float,
        **_
    ) -> NDArrayFloat:
        unscaled_power = SimpleTurbine.power(
            power_thrust_table=power_thrust_table,
            velocities=velocities,
            air_density=air_density,
        )
        scaled_power = self._compute_scaled_power(unscaled_power)
        return scaled_power

    def _compute_scaled_power(self, power: NDArrayFloat) -> NDArrayFloat:
        return self.scaling_factor * power

    def thrust_coefficient(
        self,
        power_thrust_table: dict,
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        unscaled_thrust_coefficient = SimpleTurbine.thrust_coefficient(
            power_thrust_table=power_thrust_table,
            velocities=velocities,
        )
        return unscaled_thrust_coefficient

    def axial_induction(
        self,
        power_thrust_table: dict,
        velocities: NDArrayFloat,
        **_
    ) -> NDArrayFloat:
        unscaled_axial_induction = SimpleTurbine.axial_induction(
            power_thrust_table=power_thrust_table,
            velocities=velocities,
        )
        return unscaled_axial_induction

In [ ]:
# First, run with the unscaled SimpleTurbine model for comparison
fmodel.set_operation_model(SimpleTurbine)
fmodel.run()
initial_powers = fmodel.get_turbine_powers()
print("Unscaled Powers [W]:\n", initial_powers, "\n")

# Then, run with the scaled model
fmodel.set_operation_model(ScaledTurbine(scaling_factor=1.2))
fmodel.run()

print("ScaledTurbine powers [W]:\n", fmodel.get_turbine_powers(), "\n")

## Prepackaged operation models

Naturally, prepackaged operation models can also be used in this way. In fact, we just did that with the `SimpleTurbine` model! Let's take a look at using the `CosineLossTurbine` operation model from FLORIS, either as one of the preset defaults or by passing the class in directly.

In [ ]:
from floris.core.turbine import CosineLossTurbine

fmodel.set_operation_model("simple")
fmodel.set(
    yaw_angles=np.array([[0.0, 20.0], [0.0, 20.0], [0.0, 20.0]]),
)
fmodel.run()

# Simple model does not respond to yaw angles, so powers are unaffected
print("Powers under simple model [W]:\n", fmodel.get_turbine_powers(), "\n")

# Now, switch to the cosine loss model as a built-in option
fmodel.set_operation_model("cosine-loss")
fmodel.run()

print("Powers under cosine-loss model [W]:\n", fmodel.get_turbine_powers(), "\n")

# Instead, we can pass in the class directly
fmodel.set_operation_model(CosineLossTurbine)
fmodel.run()

print("Powers under cosine-loss model (class) [W]:\n", fmodel.get_turbine_powers(), "\n")